## Extract - Leitura dos dados 

In [75]:
import pandas as pd
import os 
import chardet
from dotenv import load_dotenv
from dotenv import load_dotenv

# Carrega as variáveis do arquivo .env
# Como você está dentro da pasta "notebooks", usamos "../.env" para subir um nível
load_dotenv("../.env")
# Pega variável de ambiente
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH") 


In [76]:
# Abre o arquivo no modo binário ("rb" = read binary)
# Isso é necessário porque a detecção de encoding precisa analisar os bytes brutos,
# e não o texto já decodificado.
with open(RAW_DATA_PATH, "rb") as f:
    
    # Lê TODO o conteúdo do arquivo como uma sequência de bytes
    # e utiliza a biblioteca chardet para detectar automaticamente o encoding
    resultado = chardet.detect(f.read())
print(resultado)
# Salvando variável 
encoding = resultado["encoding"]

{'encoding': 'ascii', 'confidence': 1.0, 'language': 'en', 'mime_type': 'text/plain'}


In [77]:
#Leitura arquivo
df = pd.read_csv(RAW_DATA_PATH, encoding="latin-1", sep=",")

## Análise Exploratória de Dados (EDA) 


In [78]:
print(df.sample(5))


       InvoiceNo StockCode                        Description  Quantity  \
178265    552229     21991    BOHEMIAN COLLAGE STATIONERY SET         2   
121129    546737   15056BL            EDWARDIAN PARASOL BLACK         3   
330194    565917     21773    DECORATIVE ROSE BATHROOM BOTTLE         1   
252370    559109     82486  WOOD S/3 CABINET ANT WHITE FINISH         1   
354841    567901     82482  WOODEN PICTURE FRAME WHITE FINISH         1   

            InvoiceDate  UnitPrice  CustomerID         Country  
178265   5/6/2011 15:40       2.46         NaN  United Kingdom  
121129  3/16/2011 11:52       5.95     13358.0  United Kingdom  
330194   9/7/2011 16:15       2.46         NaN  United Kingdom  
252370   7/6/2011 11:52       8.95     15021.0  United Kingdom  
354841  9/22/2011 16:28       5.79         NaN  United Kingdom  


In [79]:
df.shape #Quantidade de linhas e colunas


(541909, 8)

In [80]:
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [81]:
df.isnull().sum().sum()

np.int64(136534)

In [82]:

df.duplicated().sum()

np.int64(5268)

In [83]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [84]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [85]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

# Tratativas

### Tratando dados str e colunas

In [86]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [88]:
### Renomeando Columns

colunas_traduzidas = {
    "InvoiceNo": "numero_fatura",
    "StockCode": "codigo_produto",
    "Description": "descricao",
    "Quantity": "quantidade",
    "InvoiceDate": "data_fatura",
    "UnitPrice": "preco_unitario",
    "CustomerID": "id_cliente",
    "Country": "pais"
}


df = df.rename(columns=colunas_traduzidas)
df.columns

Index(['numero_fatura', 'codigo_produto', 'descricao', 'quantidade',
       'data_fatura', 'preco_unitario', 'id_cliente', 'pais'],
      dtype='str')

In [89]:
### Tratamento coluna País/Country

#Unifica os valores para visualização
print(df['pais'].unique()) 

# Tratamento: Retira os espaços vázios, deixa a formatação Title e substitui os valores srt
df['pais'] = (df['pais'].str.strip().str.title().replace({
        "Eire": "Ireland",
        "Usa": "United States",
        "Rsa": "South Africa",
        "Unspecified": None,
        "European Community": "Europe"
    })) 


print(df['pais'].isnull().sum())

df = df.dropna(subset=['pais'])

df['pais'] = df['pais'].astype('category')

<StringArray>
[      'United Kingdom',               'France',            'Australia',
          'Netherlands',              'Germany',               'Norway',
                 'EIRE',          'Switzerland',                'Spain',
               'Poland',             'Portugal',                'Italy',
              'Belgium',            'Lithuania',                'Japan',
              'Iceland',      'Channel Islands',              'Denmark',
               'Cyprus',               'Sweden',              'Austria',
               'Israel',              'Finland',              'Bahrain',
               'Greece',            'Hong Kong',            'Singapore',
              'Lebanon', 'United Arab Emirates',         'Saudi Arabia',
       'Czech Republic',               'Canada',          'Unspecified',
               'Brazil',                  'USA',   'European Community',
                'Malta',                  'RSA']
Length: 38, dtype: str
446


In [90]:
### Tratamento coluna data_fatura

# Converte a coluna 'data_fatura' para o tipo datetime
# errors="coerce" garante que valores inválidos sejam transformados em NaT (Not a Time),
# evitando que o código quebre durante a execução
df['data_fatura'] = pd.to_datetime(df['data_fatura'], errors="coerce")




In [91]:
df_filtro = df[
    df["pais"].isna() |
    df["descricao"].isna() |
    df["id_cliente"].isna()
]

df_filtro.info()

<class 'pandas.DataFrame'>
Index: 134878 entries, 622 to 541540
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   numero_fatura   134878 non-null  str           
 1   codigo_produto  134878 non-null  str           
 2   descricao       133424 non-null  str           
 3   quantidade      134878 non-null  int64         
 4   data_fatura     134878 non-null  datetime64[us]
 5   preco_unitario  134878 non-null  float64       
 6   id_cliente      0 non-null       float64       
 7   pais            134878 non-null  category      
dtypes: category(1), datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 8.4 MB


In [ ]:
df_filtro.reset_index(drop=True)

df_filtro_preco = df[df['preco_unitario'] == 0]

df_filtro_preco

0         False
1         False
2         False
3         False
4         False
          ...  
541904    False
541905    False
541906    False
541907    False
541908    False
Name: preco_unitario, Length: 406585, dtype: bool

In [93]:
df = df.dropna(
    subset=["pais", "descricao", "id_cliente"],
    how="any"  # padrão, mas bom deixar explícito
)

# Outros

In [ ]:

'''
df.isnull().sum() #FAZER TRATAMENTO DOS NULLS

df = df.drop_duplicates(subset=["CustomerID"])

print(f"Quantidade de linhas: {df.shape[0]}\nQuantidade de Colunas: {df.shape[1]} ") 

df = df.drop_duplicates().reset_index(drop=True)
df 
'''

'\ndf.isnull().sum() #FAZER TRATAMENTO DOS NULLS\n\ndf = df.drop_duplicates(subset=["CustomerID"])\n\nprint(f"Quantidade de linhas: {df.shape[0]}\nQuantidade de Colunas: {df.shape[1]} ") \n\ndf = df.drop_duplicates().reset_index(drop=True)\ndf \n'

In [ ]:
df.shape

#df.info()

(406585, 8)